# Conditional TensorRT-LLM support audit
Run from this package directory in your existing Blackwell-capable kernel.


In [ ]:
from pathlib import Path
import json, sys
assert Path("launch.py").exists(), "Open the notebook inside the extracted package folder"
from launch import run, train_all, benchmarks
C = json.loads(Path("config.json").read_text())
ROOT = Path(C["run_dir"])


## Backend-dependent requirement
B200/Linux alone does not establish TensorRT-LLM support for a release and a custom-pruned Qwen checkpoint. Record the installed release and its documented supported hardware/model shapes. PyTorch remains the measured backend unless a separate engine is built and tested.

In [ ]:
import importlib.metadata, shutil, platform
from protocol import atomic_json
try:
    version = importlib.metadata.version("tensorrt_llm")
except importlib.metadata.PackageNotFoundError:
    version = None
audit = {"platform": platform.platform(), "tensorrt_llm": version, "builder": shutil.which("trtllm-build"), "status": "UNAVAILABLE_NOT_BENCHMARKED" if version is None else "INSTALLED_SUPPORT_REQUIRES_VERIFICATION", "reference": "https://nvidia.github.io/TensorRT-LLM/reference/support-matrix.html", "note": "Absence is not proof of hardware incompatibility; do not call PyTorch timing TensorRT timing."}
atomic_json(ROOT / "reports/tensorrt_support.json", audit)
print(json.dumps(audit, indent=2))

## If supported: conversion, build and benchmark
The optional helper uses your installed release’s own Qwen converter and trtllm-build. It measures conversion/build/load time, warmed repeated generation, actual generated-token counts, and the paired PPL/PIQA subset via context logits. APIs differ by release: it fails explicitly rather than claiming a fallback result. Set the two paths below and uncomment the subprocess call ONLY in a supported TensorRT environment. Keep it separate from the locked training environment.

In [ ]:
import os, subprocess
TRT_PYTHON = Path("/path/to/tensorrt/environment/bin/python")
TRT_SOURCE = Path("/path/to/matching/TensorRT-LLM/source")
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = C["gpu_uuid"]
# subprocess.run([str(TRT_PYTHON), "-u", "tensorrt_optional.py", "--source", str(TRT_SOURCE), "--run", str(ROOT), "--job", "moderate_sensitivity_s42_kl"], env=env, check=True)

## Refresh handoff
After recording support status or completing the optional benchmark, rebuild the report/ZIP to include that evidence.

In [ ]:
run("report")